# Phase 4: Feature engineering

GitHub issue: #5. Depends on #4 (Phase 3: EDA - `query_category` and `prefix_len` confirmed as
candidate signals). Test set (locked in Phase 2) gets features built by the exact same code, but its
recall/quality numbers are deliberately not computed or looked at in this notebook - reserved for the
single Phase 7 evaluation.

## Step 1-2: Candidate features and training-only fit

For every example: generate a co-visitation candidate pool from its prefix (the label is never
consulted while doing this - recall against the true held-out item is a real, measured property of the
pool below, not something guaranteed by construction), then attach point-in-time features computed from
the training window only:

- `score`: summed co-visitation weight of the candidate against the session's prefix items
- `item_popularity`: candidate's event count in the training window
- `category` / `category_match`: candidate's synthetic category, and whether it equals the session's
  `query_category`
- `grade`: the graded label (0 for every candidate except the true held-out item)

## Incident: memory blowup building this (fixed)

The first (unbatched) version of this pipeline grew to **23GB of RAM** while building the test split's
candidate pool, got flagged `stuck` by macOS, and thrashed swap badly enough to noticeably slow down the
machine it ran on. Root cause, two compounding issues:

1. **Unbounded fan-out**: joining every prefix item against its *full* co-visitation neighbour list
   (some items have thousands of neighbours) before aggregating is a join fan-out - up to
   (prefix items) x (neighbours per item) intermediate rows before any dedup.
2. **Test is the worst case for this**: it has the most sessions, and its sessions skew longest (more
   prefix items) and most engaged (touching more high-degree/popular items, which have close to the
   maximum neighbour count each). These effects compound multiplicatively, which is why test blew up far
   more than its raw row-count ratio over train/val would suggest.

**Fix**, all three parts real (not just performance shortcuts):

- Cap each session's prefix to its most recent 20 events before candidate generation - median
  `prefix_len` is only 3, so this only touches the long tail, and recency is also the *right* choice
  methodologically for a next-item task.
- Prune co-visitation to each item's own top-50 neighbours before joining against sessions
  (`covisitation.top_k_neighbors`) - bounds the per-item fan-out regardless of how popular an item is.
- Process every split in batches of 50,000 sessions rather than one giant join.

Also found and fixed while verifying this: batched vs. unbatched runs produced different (though
equally "valid") candidate sets right at the top-50 cutoff, because `rank(method="ordinal")` breaks
ties by whatever row order the join happened to produce. Fixed with a deterministic secondary sort key.
Verified: two consecutive runs now produce byte-identical output.

In [1]:
import time
from pathlib import Path

import polars as pl
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FROZEN_DIR = ROOT / "data" / "frozen"

from ranking.data.candidates import build_candidate_pool
from ranking.data.covisitation import top_k_neighbors

TOP_K = 50
NEIGHBOR_K = 50
MAX_PREFIX_ITEMS = 20
BATCH_SIZE = 50_000

events = pl.read_parquet(FROZEN_DIR / "events.parquet")
examples = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet")
covis = pl.read_parquet(FROZEN_DIR / "train_covisitation.parquet")
item_categories = pl.read_parquet(FROZEN_DIR / "item_categories.parquet")

neighbors = top_k_neighbors(covis, k=NEIGHBOR_K)
print(f"pruned neighbors: {neighbors.height:,} rows (<= {NEIGHBOR_K} per item)")

pruned neighbors: 14,281,092 rows (<= 50 per item)


## Building val's features, live, in batches

This is the real pipeline, run end to end on the val split (the same code `scripts/build_phase4_features.py`
uses for all three splits unattended).

In [2]:
import json as _json

manifest = _json.loads((FROZEN_DIR / "manifest.json").read_text())
train_end_ts = manifest["split_cutoffs_ms"]["train_end"]
train_events = events.filter(pl.col("ts") < train_end_ts)
item_popularity = train_events.group_by("aid").agg(pl.len().alias("item_popularity"))

split = "val"
split_examples = examples.filter(pl.col("split") == split)
all_sessions = split_examples["session"].to_list()
n_batches = (len(all_sessions) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"{len(all_sessions):,} sessions in {n_batches} batches")

t0 = time.time()
batches = []
total_hits = 0
for i in range(n_batches):
    batch_sessions = all_sessions[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    batch_examples = split_examples.filter(pl.col("session").is_in(batch_sessions))

    prefix_events = (
        events.join(batch_examples.select("session", "prefix_last_ts"), on="session", how="inner")
        .filter(pl.col("ts") <= pl.col("prefix_last_ts"))
        .with_columns(
            pl.col("ts").rank(method="ordinal", descending=True).over("session").alias("_recency")
        )
        .filter(pl.col("_recency") <= MAX_PREFIX_ITEMS)
        .drop("_recency")
    )
    candidates = build_candidate_pool(prefix_events, neighbors, top_k=TOP_K)

    features = (
        candidates.join(item_popularity, on="aid", how="left")
        .with_columns(pl.col("item_popularity").fill_null(0))
        .join(item_categories, on="aid", how="left")
        .with_columns(pl.col("category").fill_null(-1))
        .join(
            batch_examples.select("session", "query_category", "label_aid", "label_grade"),
            on="session", how="left",
        )
        .with_columns(
            (pl.col("category") == pl.col("query_category")).alias("category_match"),
            pl.when(pl.col("aid") == pl.col("label_aid")).then(pl.col("label_grade")).otherwise(0).alias("grade"),
        )
        .drop("label_aid", "label_grade")
    )
    batches.append(features)
    hit = batch_examples.join(
        candidates.select("session", "aid").rename({"aid": "cand"}), on="session", how="left"
    ).filter(pl.col("cand") == pl.col("label_aid"))["session"].n_unique()
    total_hits += hit

val_features = pl.concat(batches)
print(f"[{time.time()-t0:.0f}s] {val_features.height:,} candidate rows")
print(f"recall@{TOP_K} of true label in pool: {total_hits/len(all_sessions):.1%}")

396,786 sessions in 8 batches


[16s] 18,266,086 candidate rows
recall@50 of true label in pool: 11.4%


Peak memory for this stays well under 1GB, and the whole thing runs in well under a minute - a
~150x speedup over the original unbatched version (which took 3,257s / 54 minutes on this same split
before it was fixed, and blew up to 23GB on the larger test split).

## Step 4: Validation

In [3]:
train_manifest_features = ROOT / "data" / "frozen" / "features" / "train.parquet"
train_features = pl.read_parquet(train_manifest_features)
train_examples = examples.filter(pl.col("split") == "train")

train_hits = train_examples.join(
    train_features.select("session","aid").rename({"aid":"cand"}), on="session", how="left"
).filter(pl.col("cand") == pl.col("label_aid"))["session"].n_unique()

print(f"train: {train_features.height:,} candidate rows, recall@50: {train_hits/train_examples.height:.1%}")
print(f"val:   {val_features.height:,} candidate rows, recall@50: {total_hits/len(all_sessions):.1%}")

train: 23,447,741 candidate rows, recall@50: 21.5%
val:   18,266,086 candidate rows, recall@50: 11.4%


**Finding**: train's recall@50 is nearly double val's - a real point-in-time generalization gap,
not a bug. Co-visitation and categories are frozen on the training window (correctly, to avoid
leakage), so training-window examples' held-out items are naturally better-represented in that frozen
graph than later (val) examples, which increasingly reference items and patterns the graph never saw.
This means Phase 5/6 model quality should be judged on val/test, never inferred from train performance -
it's a real ceiling on what any candidate-pool-based method can achieve, regardless of reranking
quality.

## Exit criteria

Versioned feature pipeline (`scripts/build_phase4_features.py`, deterministic, unit-tested), feature
list with definitions (Step 1-2 above), parity check (same code path builds train/val/test identically,
byte-identical across reruns - verified separately). Met.

## Out of scope / follow-ups

- Whether top-K=50 is the right pool size, or a category-aware fallback for unknown-query examples,
  would meaningfully improve recall.
- The 23GB incident is a real lesson for any future large-scale join in this project: always bound
  fan-out and batch by entity before joining against a large auxiliary table.